# General Health Query Chatbot
### Task 4 — DevelopersHub Corp AI/ML Internship

**Model:** Llama 3 (llama-3.1-8b-instant) via Groq API  
**Focus:** Prompt Engineering + Safety Filtering


## Step 1 — Imports

In [ ]:
import requests

print("Libraries imported successfully!")


Libraries imported successfully!


## Step 2 — Configuration

In [ ]:
# API TOKEN Configuration
GROQ_API_KEY = "" # Replace with your actual Groq API key - due to some security reasons, the key is not included here. You must set it in your environment or directly in the code.

# check the screenshots for more information

API_URL = "https://api.groq.com/openai/v1/chat/completions"

print("Configuration set!")


Configuration set!


## Step 3 — System Prompt (Prompt Engineering)

In [47]:
# System prompt to guide AI model behavior

SYSTEM_PROMPT = """You are a helpful and friendly medical information assistant.
Your job is to answer general health questions in simple, clear language that anyone can understand.

IMPORTANT RULES you must always follow:
- Always recommend seeing a doctor for serious, persistent, or emergency symptoms.
- Never diagnose any disease or medical condition.
- Never prescribe or recommend specific medications or dosages.
- Keep answers concise: 3 to 5 sentences maximum.
- Use simple, everyday language, and avoid complex medical jargon.
- If a question asks about something dangerous, illegal, or harmful, politely decline.
- Always end every response with exactly this line:
  "Please consult a healthcare professional for personalized medical advice."

Your tone should be warm, calm, and reassuring like a knowledgeable friend, not a cold textbook."""

print("System prompt configured!")


System prompt configured!


## Step 4 — Safety Filter

In [52]:
# Safety keywords to block harmful queries

BLOCKED_KEYWORDS = [
    "overdose",
    "self-harm",
    "self harm",
    "suicide",
    "kill myself",
    "illegal drug",
    "how to harm",
    "poison someone",
]

def safety_filter(user_question: str) -> str | None:
    """
    Check if the user's question contains any blocked content.

    Returns:
        A safe refusal message (str) if blocked.
        None if the question is safe to proceed.
    """
    question_lower = user_question.lower()
    for keyword in BLOCKED_KEYWORDS:
        if keyword in question_lower:
            return (
                "I'm not able to assist with that query. "
                "If you or someone you know is in distress, please reach out to a "
                "healthcare professional or a mental health helpline immediately."
            )
    return None  # Safe to proceed

print("Safety filter ready!")


Safety filter ready!


## Step 5 — Main Chatbot Function

In [53]:
# Main chatbot function - runs filters, builds prompt and sends request to Groq.

def ask_health_question(user_question: str) -> str:
    """
    Send a health question to Llama 3 and return the response.

    Args:
        user_question: The health question typed by the user.

    Returns:
        The AI's response as a clean string.
    """
    # safety check
    blocked_response = safety_filter(user_question)
    if blocked_response:
        return blocked_response

    # prepare the API request
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": "llama-3.1-8b-instant",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_question},
        ],
        "temperature": 0.4,  # lower provides more factual responses
        "max_tokens": 250,   # AI response maximum length
    }

    # make the API call
    try:
        response = requests.post(API_URL, headers=headers, json=payload, timeout=30)

        if response.status_code == 200:
            return response.json()["choices"][0]["message"]["content"].strip()
        else:
            return f"API Error {response.status_code}: {response.text[:200]}"

    except requests.exceptions.Timeout:
        return "Request timed out. Please try again."
    except requests.exceptions.RequestException as e:
        return f"Connection error: {str(e)}"

print("ask_health_question() function defined and ready!")


ask_health_question() function defined and ready!


## Step 6 — Required Test Queries

In [ ]:
# import textwrap  - for text wrapping

required_questions = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?",
]

print("=" * 65)
print("REQUIRED TEST QUERIES")
print("=" * 65)

for i, question in enumerate(required_questions, 1):
    print(f"\n[Q{i}] {question}")
    print("-" * 50)
    answer = ask_health_question(question)
    # wrapped = textwrap.fill(answer, width=150)
    print(f"{answer}")
    print()

REQUIRED TEST QUERIES

[Q1] What causes a sore throat?
--------------------------------------------------
A sore throat can be caused by a variety of things, including a viral infection like a cold or flu, allergies, or irritation from smoke, dust, or dry air. It can also be caused by a bacterial infection like strep throat. Sometimes, a sore throat can be a sign of a more serious underlying condition, so it's always best to see a doctor if you're experiencing persistent or severe symptoms. 

Please consult a healthcare professional for personalized medical advice.


[Q2] Is paracetamol safe for children?
--------------------------------------------------
Paracetamol can be used to relieve pain and reduce fever in children, but it's essential to use it carefully and follow the recommended dosage. Always check the packaging for the correct dosage for your child's age and weight. It's also crucial to consult with your child's doctor or pharmacist before giving paracetamol to a child, esp

## Step 7 — Additional Test Queries

In [66]:
additional_questions = [
    "How much water should I drink per day?",
    "What are the symptoms of dehydration?",
    "How can I reduce a mild fever at home?",
    "What foods are good for a healthy heart?",
    "Is it normal to feel tired all the time?",
]

print("=" * 65)
print("ADDITIONAL TEST QUERIES")
print("=" * 65)

for i, question in enumerate(additional_questions, 1):
    print(f"\n[Q{i+2}] {question}")
    print("-" * 50)
    answer = ask_health_question(question)
    # wrapped = textwrap.fill(answer, width=150)
    print(f"{answer}")
    print()


ADDITIONAL TEST QUERIES

[Q3] How much water should I drink per day?
--------------------------------------------------
The amount of water you should drink each day varies, but a general guideline is to aim for at least 8-10 cups (64-80 ounces) of water per day. However, this can depend on your age, sex, weight, activity level, and climate. For example, athletes or people living in hot, humid areas may need more water to stay hydrated.

It's also worth noting that you can get some of your daily water intake from other fluids, like milk, juice, and tea, as well as from foods that are high in water content, like fruits and vegetables.

Please consult a healthcare professional for personalized medical advice.


[Q4] What are the symptoms of dehydration?
--------------------------------------------------
Dehydration symptoms can be mild or severe. Common signs include dry mouth, dark yellow or brown urine, dizziness or lightheadedness, headaches, fatigue, and thirst. In severe cases, dehy

## Step 8 — Safety Filter Test

In [68]:
harmful_queries = [
    "How do I overdose on medication?",
    "I want to self-harm, what should I do?",
]

print("=" * 65)
print("SAFETY FILTER TESTS (These should be BLOCKED)")
print("=" * 65)

for query in harmful_queries:
    print(f"\nQuery: {query}")
    print("-" * 50)
    result = ask_health_question(query)
    print(f"{result}")
    print()

print("Safety filter working correctly!")


SAFETY FILTER TESTS (These should be BLOCKED)

Query: How do I overdose on medication?
--------------------------------------------------
I'm not able to assist with that query. If you or someone you know is in distress, please reach out to a healthcare professional or a mental health helpline immediately.


Query: I want to self-harm, what should I do?
--------------------------------------------------
I'm not able to assist with that query. If you or someone you know is in distress, please reach out to a healthcare professional or a mental health helpline immediately.

Safety filter working correctly!


## Step 9 — Interactive Mode

Type your own questions. Type `quit` to stop.

In [ ]:
# Interactive loop — type 'quit' or 'exit' to stop

print("HEALTH QUERY CHATBOT — Interactive Mode")
print("Type your health question and press Enter.")
print("Type 'quit' or 'exit' to stop.")
print("=" * 50)

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue

    if user_input.lower() in ["quit", "exit", "bye", "q"]:
        print("Goodbye! Stay healthy!")
        break

    response = ask_health_question(user_input)
    print(f"\n{response}\n")
    print("-" * 50)


HEALTH QUERY CHATBOT — Interactive Mode
Type your health question and press Enter.
Type 'quit' or 'exit' to stop.
Goodbye! Stay healthy!
